# Stock Opportunity Forecaster: Colab quickstart

Runs the offline demo and evaluation (no keys needed), then the live review, live evaluation and scanner backtest with real public data and real models.

**Research and education only. Not financial advice. Paper trading only.**

## 1. Get the code
Clones the public repository.

In [ ]:
!git clone https://github.com/kotimaddala/stock-forecaster.git
%cd stock-forecaster
!pip install -q -r requirements.txt yfinance langgraph

## 2. Offline: tests, demo and evaluation (no keys)

In [ ]:
!python -m pytest -q
!python -m forecaster.cli demo
!python -m eval.run_eval

In [ ]:
from IPython.display import Markdown
Markdown(open('eval/results/offline_results.md').read())

## 3. Keys for live mode
Add these in Colab's **Secrets** panel (key icon on the left), not in a cell:
- `OPENROUTER_API_KEY`
- `SEC_USER_AGENT` (your name and email, required by the SEC)

Optional: `ANALYST_MODEL`, `REVIEWER_MODEL` (the reviewer should be a different, stronger model).

In [ ]:
import os
from google.colab import userdata
for k in ['OPENROUTER_API_KEY', 'SEC_USER_AGENT', 'ANALYST_MODEL', 'REVIEWER_MODEL']:
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception:
        pass
print({k: bool(os.environ.get(k)) for k in ['OPENROUTER_API_KEY', 'SEC_USER_AGENT']})

## 4. Live thesis review on public data
Pulls the latest 10-Q/10-K/8-K text and XBRL figures from SEC EDGAR, prices, and Federal Register documents, then runs the multi-agent review. Edit `examples/thesis_example.json` for your own thesis.

In [ ]:
!python -m forecaster.cli review examples/thesis_example.json --trigger "Latest quarterly filing"

In [ ]:
# Same inputs, single chain-of-thought baseline
!python -m forecaster.cli review examples/thesis_example.json --trigger "Latest quarterly filing" --mode single

## 5. Live evaluation: Tree of Thoughts vs single chain of thought
Runs real models on the fixed scenarios. Start with `--repeats 1` to check cost (about 5-10 calls per ToT review).

In [ ]:
!python -m eval.run_live_eval --repeats 3
Markdown(open('eval/results/live_results.md').read())

## 6. Scanner backtest (decision outcomes)
Point-in-time: each day the scanner sees only past data. Compared with SPY over the same holding periods.

In [ ]:
!python -m eval.backtest_scanner --start 2024-01-01
Markdown(open('eval/results/backtest_results.md').read())

## 7. Commit results back to the repository
After the runs, commit `eval/results/` so reviewers can see them.